In [1]:
# ============================================================
# R1-26  Duration / volume control  (SSHO-D-26-05807)
# Input: temporal_edges_with_community3.csv  (event-level, 21,420 rows)
# ============================================================
import pandas as pd, numpy as np, datetime as dt
from scipy.stats import chi2_contingency, mannwhitneyu

# --- Colab upload (uncomment if not already in session) ---
# from google.colab import files; files.upload()

FILE = "temporal_edges_with_community3.csv"
df = pd.read_csv(FILE, sep=";", encoding="utf-8-sig")

# robust parse
df["ts"]   = pd.to_datetime(df["date"],     format="%d/%m/%Y %H:%M", errors="coerce")
df["day"]  = pd.to_datetime(df["date_day"], format="%d/%m/%Y",       errors="coerce").dt.date
df["cross"] = df["is_cross_community"].astype(str).str.upper().eq("TRUE").astype(int)

print("rows:", len(df), "| date range:", df["day"].min(), "->", df["day"].max())
print("NaT date:", df["ts"].isna().sum(), "| NaT day:", df["day"].isna().sum())
print("cross events (should be 3258):", df["cross"].sum())
print("="*60)

# ---------- helpers ----------
def chi_2x2(pre, post, correction):
    a = [pre["cross"].sum(),  len(pre)  - pre["cross"].sum()]
    b = [post["cross"].sum(), len(post) - post["cross"].sum()]
    chi2, p, dof, _ = chi2_contingency([a, b], correction=correction)
    N = len(pre) + len(post)
    V = np.sqrt(chi2 / N)                      # 2x2 -> min(r-1,c-1)=1
    return chi2, p, V

def line(name, pre, post):
    pr = 100*pre["cross"].mean()
    po = 100*post["cross"].mean()
    c0,p0,v0 = chi_2x2(pre, post, correction=False)  # Pearson
    c1,p1,_  = chi_2x2(pre, post, correction=True)   # Yates
    print(f"{name}")
    print(f"  pre : {pre['cross'].sum():>5}/{len(pre):<6} = {pr:5.2f}%   (n_days {pre['day'].nunique()})")
    print(f"  post: {post['cross'].sum():>5}/{len(post):<6} = {po:5.2f}%   (n_days {post['day'].nunique()})")
    print(f"  chi2(Pearson)={c0:6.2f} p={p0:.2e} | chi2(Yates)={c1:6.2f} p={p1:.2e} | Cramer V={v0:.3f}")
    print()

# ---------- CHECKPOINT: reproduce Table 3 ----------
d1  = dt.date(2025,8,1);  d26 = dt.date(2025,8,26)
d27 = dt.date(2025,8,27); d28 = dt.date(2025,8,28); d29 = dt.date(2025,9,29)

pre_strict  = df[(df["day"]>=d1)  & (df["day"]<=d26)]
post_strict = df[(df["day"]>=d28) & (df["day"]<=d29)]      # Aug 27 EXCLUDED (matches Methods text)
post_incl27 = df[(df["day"]>=d27) & (df["day"]<=d29)]      # Aug 27 IN post   (matches phase col / Table 3)
aug27       = df[df["day"]==d27]

print("CHECKPOINT vs Table 3  (manuscript prints pre 13.5% / post 16.9%, chi2=45.58)")
print(f"Aug 27 events folded into post by the phase column: {len(aug27)}")
print("-"*60)
line("(A) post = Aug 27-Sep 29  [phase column / current Table 3]", pre_strict, post_incl27)
line("(B) post = Aug 28-Sep 29  [strict, Aug 27 excluded]",        pre_strict, post_strict)
print("="*60)

# ============================================================
# ROBUSTNESS A  -  matched-length windows around Aug 27
# Equalizes duration on both sides; excludes Aug 27 by construction.
# ============================================================
print("ROBUSTNESS A  |  matched-length windows (pre = last N days <=Aug26, post = first N days >=Aug28)")
print("-"*60)
for N in (7, 14, 21):
    pre_w  = df[(df["day"]>=d26-dt.timedelta(days=N-1)) & (df["day"]<=d26)]
    post_w = df[(df["day"]>=d28) & (df["day"]<=d28+dt.timedelta(days=N-1))]
    line(f"N = {N} days per side", pre_w, post_w)

# ============================================================
# ROBUSTNESS B  -  daily-rate test (unit = day, volume-invariant)
# ============================================================
print("="*60)
print("ROBUSTNESS B  |  daily cross-community ratio, Mann-Whitney (day as unit)")
print("-"*60)

daily = (df.dropna(subset=["day"])
           .groupby("day")
           .agg(cross=("cross","sum"), total=("cross","size")))
daily["prop"] = daily["cross"] / daily["total"]

def mwu_report(tag, post_days, pre_days):
    n1, n2 = len(post_days), len(pre_days)
    U, p_g = mannwhitneyu(post_days, pre_days, alternative="greater")
    _, p_t = mannwhitneyu(post_days, pre_days, alternative="two-sided")
    cles = U/(n1*n2)          # P(random post-day > random pre-day)
    rb   = 2*cles - 1         # rank-biserial
    rng = np.random.default_rng(42)
    boot = [rng.choice(post_days,n1,True).mean() - rng.choice(pre_days,n2,True).mean()
            for _ in range(10000)]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{tag}")
    print(f"  pre : n_days={n2}  median={np.median(pre_days):.3f}  mean={np.mean(pre_days):.3f}")
    print(f"  post: n_days={n1}  median={np.median(post_days):.3f}  mean={np.mean(post_days):.3f}")
    print(f"  U={U:.0f}  p(greater)={p_g:.4f}  p(two-sided)={p_t:.4f}")
    print(f"  rank-biserial r={rb:.3f}  CLES={cles:.3f}")
    print(f"  mean daily diff (post-pre)={np.mean(post_days)-np.mean(pre_days):+.3f}  "
          f"95% boot CI [{lo:+.3f}, {hi:+.3f}]  (B=10000, seed 42)")
    print()

pre_days     = daily.loc[[d for d in daily.index if d1  <= d <= d26], "prop"].values
post_days_st = daily.loc[[d for d in daily.index if d28 <= d <= d29], "prop"].values
post_days_27 = daily.loc[[d for d in daily.index if d27 <= d <= d29], "prop"].values

mwu_report("(B) post = Aug 28-Sep 29  [strict]",        post_days_st, pre_days)
mwu_report("(A) post = Aug 27-Sep 29  [Aug 27 in post]", post_days_27, pre_days)

# ============================================================
# FLAG 2 note  -  source_community corruption (repo hygiene, R1-19)
# ============================================================
sc = pd.to_numeric(df["source_community"], errors="coerce")
print("="*60)
print("FLAG 2  source_community is true_community x10:",
      bool((sc.dropna() % 10 == 0).all()),
      "| divide by 10 before any public-repo release (R1-19)")

rows: 21420 | date range: 2025-08-01 -> 2025-09-29
NaT date: 0 | NaT day: 0
cross events (should be 3258): 3258
CHECKPOINT vs Table 3  (manuscript prints pre 13.5% / post 16.9%, chi2=45.58)
Aug 27 events folded into post by the phase column: 406
------------------------------------------------------------
(A) post = Aug 27-Sep 29  [phase column / current Table 3]
  pre :  1445/10670  = 13.54%   (n_days 26)
  post:  1813/10750  = 16.87%   (n_days 34)
  chi2(Pearson)= 45.84 p=1.29e-11 | chi2(Yates)= 45.58 p=1.47e-11 | Cramer V=0.046

(B) post = Aug 28-Sep 29  [strict, Aug 27 excluded]
  pre :  1445/10670  = 13.54%   (n_days 26)
  post:  1726/10344  = 16.69%   (n_days 33)
  chi2(Pearson)= 40.50 p=1.96e-10 | chi2(Yates)= 40.26 p=2.23e-10 | Cramer V=0.044

ROBUSTNESS A  |  matched-length windows (pre = last N days <=Aug26, post = first N days >=Aug28)
------------------------------------------------------------
N = 7 days per side
  pre :   802/4777   = 16.79%   (n_days 7)
  post:  1305/877